In [9]:
# ##translate UI elements

# # --- Imports ---
# import json
# import os
# from pathlib import Path
# import asyncio
# from openai import AsyncAzureOpenAI
# from dotenv import load_dotenv


# # --- Load .env ---
# load_dotenv()

# # --- Configuration ---
# INPUT_JSON = "../03_Outputs/UI_Translations/en.json"
# OUTPUT_FOLDER = "../03_Outputs/UI_Translations"
# TARGET_LANGUAGES = [ "zh", "ru"]#"fr", "es", "pt", "ar",

# AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
# AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
# AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
# AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

# # --- Translator Client ---
# client = AsyncAzureOpenAI(
#     api_key=AZURE_OPENAI_API_KEY,
#     azure_endpoint=AZURE_OPENAI_ENDPOINT,
#     api_version=AZURE_OPENAI_API_VERSION,
# )

# # --- Load JSON ---
# def load_json(path):
#     with open(path, "r", encoding="utf-8") as f:
#         return json.load(f)

# def save_json(data, path):
#     Path(path).parent.mkdir(parents=True, exist_ok=True)
#     with open(path, "w", encoding="utf-8") as f:
#         json.dump(data, f, ensure_ascii=False, indent=2)

# # --- Flattening ---
# def flatten_json(data, prefix=""):
#     items = {}
#     if isinstance(data, dict):
#         for k, v in data.items():
#             new_key = f"{prefix}.{k}" if prefix else k
#             items.update(flatten_json(v, new_key))
#     elif isinstance(data, list):
#         for i, v in enumerate(data):
#             new_key = f"{prefix}[{i}]"
#             items.update(flatten_json(v, new_key))
#     else:
#         items[prefix] = data
#     return items

# # --- Unflattening ---
# def unflatten_json(flat):
#     root = {}
#     for compound_key, value in flat.items():
#         keys = compound_key.replace("]", "").replace("[", ".").split(".")
#         ref = root
#         for key in keys[:-1]:
#             if key.isdigit():
#                 key = int(key)
#                 while len(ref) <= key:
#                     ref.append({})
#                 ref = ref[key]
#             else:
#                 ref = ref.setdefault(key, {})
#         last_key = keys[-1]
#         if last_key.isdigit():
#             last_key = int(last_key)
#             while len(ref) <= last_key:
#                 ref.append(None)
#             ref[last_key] = value
#         else:
#             ref[last_key] = value
#     return root

# # --- Translate One ---
# async def translate_text(text):
#     prompt = (
#         "You are a professional translator. Preserve any HTML tags in the input. "
#         "Translate this text into JSON format as follows: "
#         "{\"zh\": \"\", \"ru\": \"\"}"#\"fr\": \"\", \"es\": \"\", \"pt\": \"\", \"ar\": \"\", 
#         f"\n\nText: {text}"
#     )
#     response = await client.chat.completions.create(
#         model=AZURE_OPENAI_DEPLOYMENT,
#         messages=[
#             {"role": "system", "content": "You are a multilingual translator that outputs only valid JSON."},
#             {"role": "user", "content": prompt}
#         ],
#         temperature=0,
#         max_tokens=1000
#     )
#     return json.loads(response.choices[0].message.content.strip())

# # --- Translate All ---
# async def translate_all_strings():
#     en_data = load_json(INPUT_JSON)
#     flat = flatten_json(en_data)
#     translations = {lang: {} for lang in TARGET_LANGUAGES}

#     for key, value in flat.items():
#         if not isinstance(value, str):
#             continue
#         print(f"Translating key: {key}")
#         try:
#             result = await translate_text(value)
#             for lang in TARGET_LANGUAGES:
#                 translations[lang][key] = result.get(lang, "needs translation")
#         except Exception as e:
#             print(f"⚠️ Translation failed for key: {key} — {e}")
#             for lang in TARGET_LANGUAGES:
#                 translations[lang][key] = "needs translation"

#     for lang in TARGET_LANGUAGES:
#         unflat = unflatten_json(translations[lang])
#         save_json(unflat, f"{OUTPUT_FOLDER}/{lang}.json")
#     print("✅ Translations complete.")



# await translate_all_strings()


In [1]:
## translate UI elements (run pipeline in language groups)

# --- Imports ---
import json
import os
from pathlib import Path
from openai import AsyncAzureOpenAI
from dotenv import load_dotenv

# --- Load .env ---
load_dotenv()

# --- Configuration ---
INPUT_JSON = "../03_Outputs/UI_Translations/UI/en.json"
OUTPUT_FOLDER = "../03_Outputs/UI_Translations/UI"

LANGUAGE_RUNS = [
    ["zh", "ru", "fr"],
    ["es", "pt", "ar"],
]

LANGUAGE_NAMES = {
    "zh": "Chinese",
    "ru": "Russian",
    "fr": "French",
    "es": "Spanish",
    "pt": "Portuguese",
    "ar": "Arabic",
}

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

# --- Translator Client ---
client = AsyncAzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

# --- Load / Save JSON ---
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(data, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

# --- Prompt helpers ---
def build_language_instruction(target_languages):
    return ", ".join(LANGUAGE_NAMES.get(code, code) for code in target_languages)

def build_json_schema_example(target_languages):
    schema = {lang: f"<{LANGUAGE_NAMES.get(lang, lang)} translation>" for lang in target_languages}
    return json.dumps(schema, ensure_ascii=False, indent=2)

# --- Translate a single string for one language set ---
async def translate_text(text: str, target_languages: list[str]) -> dict:
    language_list = build_language_instruction(target_languages)
    schema_example = build_json_schema_example(target_languages)

    prompt = (
        "You are a professional UI translator.\n"
        "Translate the following user interface text into the requested target languages.\n"
        "Preserve any HTML/XML tags exactly as written.\n"
        "Preserve placeholders, punctuation, spacing, and line breaks.\n"
        "Keep the wording concise and natural for a digital learning platform.\n"
        "Return ONLY valid JSON with exactly these keys:\n"
        f"{schema_example}\n\n"
        f"Target languages: {language_list}\n"
        f'Text: "{text}"'
    )

    try:
        response = await client.chat.completions.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a multilingual translator that outputs only valid JSON. "
                        "Do not use markdown fences."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            temperature=0,
            max_tokens=500,
        )

        result = response.choices[0].message.content.strip()
        print(f"🔎 Raw model output for '{text[:80]}...': {result}")

        parsed = json.loads(result)

        normalized = {}
        for lang in target_languages:
            value = parsed.get(lang, "needs translation")
            normalized[lang] = value if isinstance(value, str) else "needs translation"

        return normalized

    except Exception as e:
        print(f"⚠️ Translation failed for '{text}': {e}")
        return {lang: "needs translation" for lang in target_languages}

# --- Recursive translation walker ---
async def translate_structure(obj, target_languages):
    if isinstance(obj, dict):
        return {k: await translate_structure(v, target_languages) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [await translate_structure(v, target_languages) for v in obj]
    elif isinstance(obj, str):
        print(f"🔤 Translating: {obj[:60]}...")
        return await translate_text(obj, target_languages)
    else:
        return obj

# --- Split per language ---
def split_per_language(translated_obj, langs):
    def is_translation_block(obj):
        return (
            isinstance(obj, dict)
            and all(lang in obj for lang in langs)
            and all(isinstance(obj[lang], str) for lang in langs)
        )

    def extract(obj, lang):
        if is_translation_block(obj):
            return obj[lang]
        elif isinstance(obj, dict):
            return {k: extract(v, lang) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [extract(v, lang) for v in obj]
        else:
            return obj

    return {lang: extract(translated_obj, lang) for lang in langs}

# --- Run one language group ---
async def translate_language_group(target_languages):
    print(f"\n🚀 Running translation group: {target_languages}")
    en_data = load_json(INPUT_JSON)

    translated_full = await translate_structure(en_data, target_languages)
    per_lang = split_per_language(translated_full, target_languages)

    for lang, data in per_lang.items():
        out_path = f"{OUTPUT_FOLDER}/{lang}.json"
        save_json(data, out_path)
        print(f"✅ Saved {out_path}")

# --- Run all groups sequentially ---
async def translate_all_language_runs():
    for language_group in LANGUAGE_RUNS:
        await translate_language_group(language_group)

# --- Run ---
await translate_all_language_runs()


🚀 Running translation group: ['zh', 'ru', 'fr']
🔤 Translating: Not started...
🔎 Raw model output for 'Not started...': {
  "zh": "未开始",
  "ru": "Не начато",
  "fr": "Non commencé"
}
🔤 Translating: In progress...
🔎 Raw model output for 'In progress...': {
  "zh": "进行中",
  "ru": "В процессе",
  "fr": "En cours"
}
🔤 Translating: Completed...
🔎 Raw model output for 'Completed...': {
  "zh": "已完成",
  "ru": "Завершено",
  "fr": "Terminé"
}
🔤 Translating: Nice to see you again...
🔎 Raw model output for 'Nice to see you again...': {
  "zh": "很高兴再次见到你",
  "ru": "Рад снова видеть вас",
  "fr": "Content de vous revoir"
}
🔤 Translating: Don’t have an account yet?...
🔎 Raw model output for 'Don’t have an account yet?...': {
  "zh": "还没有账户？",
  "ru": "Нет аккаунта?",
  "fr": "Pas encore de compte ?"
}
🔤 Translating: Sign up now...
🔎 Raw model output for 'Sign up now...': {
  "zh": "立即注册",
  "ru": "Зарегистрироваться сейчас",
  "fr": "Inscrivez-vous maintenant"
}
🔤 Translating: Login...
🔎 Raw model 

In [11]:

# import json
# import os
# from pathlib import Path
# import asyncio
# from openai import AsyncAzureOpenAI
# from dotenv import load_dotenv

# # --- Load .env ---
# load_dotenv()

# # --- Config ---
# INPUT_JSON = "../03_Outputs/UI_Translations/kg_en.json"
# OUTPUT_FOLDER = "../03_Outputs/UI_Translations"
# TARGET_LANGUAGES = ["zh", "ru"]#"fr", "es", "pt", "ar", 

# AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
# AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
# AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
# AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

# client = AsyncAzureOpenAI(
#     api_key=AZURE_OPENAI_API_KEY,
#     azure_endpoint=AZURE_OPENAI_ENDPOINT,
#     api_version=AZURE_OPENAI_API_VERSION,
# )

# def load_json(path):
#     with open(path, "r", encoding="utf-8") as f:
#         return json.load(f)

# def save_json(data, path):
#     Path(path).parent.mkdir(parents=True, exist_ok=True)
#     with open(path, "w", encoding="utf-8") as f:
#         json.dump(data, f, ensure_ascii=False, indent=2)

# async def translate_text(text: str) -> dict:
#     prompt = (
#         "You are a professional translator. Preserve any HTML tags in the input. "
#         "Translate this text into JSON format as follows: "
#         "{\"fr\": \"\", \"es\": \"\", \"pt\": \"\", \"ar\": \"\", \"zh\": \"\", \"ru\": \"\"}"
#         f"\n\nText: {text}"
#     )
#     response = await client.chat.completions.create(
#         model=AZURE_OPENAI_DEPLOYMENT,
#         messages=[
#             {"role": "system", "content": "You are a multilingual translator that outputs only valid JSON."},
#             {"role": "user", "content": prompt}
#         ],
#         temperature=0,
#         max_tokens=1000
#     )
#     return json.loads(response.choices[0].message.content.strip())

# # Translate every string and wrap into per-language dict
# async def translate_structure(obj):
#     if isinstance(obj, dict):
#         return {k: await translate_structure(v) for k, v in obj.items()}
#     elif isinstance(obj, list):
#         return [await translate_structure(v) for v in obj]
#     elif isinstance(obj, str):
#         try:
#             print(f"🔤 Translating: {obj[:60]}...")
#             result = await translate_text(obj)
#             return {lang: result.get(lang, "needs translation") for lang in TARGET_LANGUAGES}
#         except Exception as e:
#             print(f"⚠️ Translation failed: {e}")
#             return {lang: "needs translation" for lang in TARGET_LANGUAGES}
#     else:
#         return obj  # pass through non-strings (e.g. ints)

# # Separate full multi-lang object into one per language
# def extract_per_language(lang_keys, translated_obj):
#     def extract(obj):
#         if isinstance(obj, dict):
#             # If this dict is a direct translation result, collapse it
#             if all(lang in obj for lang in lang_keys):
#                 return {lang: obj[lang] for lang in lang_keys}
#             return {k: extract(v) for k, v in obj.items()}
#         elif isinstance(obj, list):
#             return [extract(v) for v in obj]
#         else:
#             return obj

#     separated = {lang: {} for lang in lang_keys}

#     def walk(src, dests):
#         if isinstance(src, dict):
#             for k, v in src.items():
#                 if isinstance(v, dict) and set(v.keys()) >= set(lang_keys):
#                     for lang in lang_keys:
#                         dests[lang][k] = v[lang]
#                 else:
#                     for lang in lang_keys:
#                         dests[lang][k] = {} if isinstance(v, dict) else []
#                     walk(v, {lang: dests[lang][k] for lang in lang_keys})
#         elif isinstance(src, list):
#             for i, v in enumerate(src):
#                 if isinstance(v, dict) and set(v.keys()) >= set(lang_keys):
#                     for lang in lang_keys:
#                         dests[lang].append(v[lang])
#                 else:
#                     for lang in lang_keys:
#                         dests[lang].append({} if isinstance(v, dict) else [])
#                     walk(v, {lang: dests[lang][i] for lang in lang_keys})

#     top_level = extract(translated_obj)
#     walk(top_level, separated)
#     return separated

# # Main entry
# async def main():
#     original = load_json(INPUT_JSON)
#     translated_full = await translate_structure(original)
#     per_lang = extract_per_language(TARGET_LANGUAGES, translated_full)

#     for lang, data in per_lang.items():
#         save_json(data, f"{OUTPUT_FOLDER}/kg--{lang}.json")
#         print(f"✅ Saved {lang}.json")

# # --- Run ---
# await main()



In [4]:
import os
import json
import asyncio
from pathlib import Path
from copy import deepcopy
from openai import AsyncAzureOpenAI
from dotenv import load_dotenv

# --- Load environment ---
load_dotenv()

# --- Config ---
INPUT_JSON = "../03_Outputs/SEA_Modules/en/module_structure.json"
OUTPUT_DIR = "../03_Outputs/SEA_Modules"
TARGET_LANGUAGES = ["fr", "es", "pt", "ar", "zh", "ru"]

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

client = AsyncAzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

# --- Load and save helpers ---
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(data, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

# --- Step 1: collect all translatable strings ---
def collect_strings(node, keys=("title", "description", "tooltip"), collected=None):
    if collected is None:
        collected = set()
    if isinstance(node, dict):
        for k, v in node.items():
            if k in keys and isinstance(v, str):
                cleaned = v.strip().removeprefix("[en]").strip()
                collected.add(cleaned)
            elif isinstance(v, (dict, list)):
                collect_strings(v, keys, collected)
    elif isinstance(node, list):
        for item in node:
            collect_strings(item, keys, collected)
    return collected

# --- Step 2: translate each string once for all languages ---
import re

async def translate_text(text, retries=2):
    prompt = (
        "You are a professional translator. Preserve any HTML tags in the input. "
        "Translate this text into JSON format as follows:\n"
        "{\"fr\": \"\", \"es\": \"\", \"pt\": \"\", \"ar\": \"\", \"zh\": \"\", \"ru\": \"\"}\n\n"
        "Text: " + text
    )

    for attempt in range(retries + 1):
        response = await client.chat.completions.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            messages=[
                {"role": "system", "content": "You are a multilingual translator that outputs only valid JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=1500
        )
        content = response.choices[0].message.content.strip()

        # Try parsing the content directly
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            # Attempt to extract JSON-like content using regex
            try:
                match = re.search(r'\{.*\}', content, re.DOTALL)
                if match:
                    return json.loads(match.group())
            except json.JSONDecodeError:
                pass

            # Retry with modified prompt on next iteration if available
            print(f"⚠️ Failed to parse translation (attempt {attempt+1}): {text[:80]}...")
            if attempt == retries:
                raise ValueError(f"❌ Could not parse translation for text: {text[:100]}.\nResponse was:\n{content}")


# --- Step 3: rebuild the full JSON for each language using the translation cache ---
async def rebuild_translated_json(node, lang, translations, keys=("title", "description", "tooltip")):
    if isinstance(node, dict):
        return {
            k: await rebuild_translated_json(v, lang, translations, keys)
            if isinstance(v, (dict, list)) else (
                translations[v.strip().removeprefix("[en]").strip()][lang]
                if k in keys and isinstance(v, str) else v
            )
            for k, v in node.items()
        }
    elif isinstance(node, list):
        return [await rebuild_translated_json(item, lang, translations, keys) for item in node]
    else:
        return node

# --- Main async routine ---
async def main():
    original = load_json(INPUT_JSON)
    unique_strings = collect_strings(original)
    print(f"📦 Found {len(unique_strings)} unique strings to translate")

    # Translate all strings
    translations = {}
    for text in unique_strings:
        print(f"🌍 Translating: {text[:60]}...")
        translations[text] = await translate_text(text)

    # For each language, generate the translated version
    for lang in TARGET_LANGUAGES:
        print(f"🛠 Building translated structure for: {lang}")
        translated = await rebuild_translated_json(deepcopy(original), lang, translations)
        out_path = os.path.join(OUTPUT_DIR, lang, "module_structure.json")
        save_json(translated, out_path)
        print(f"✅ Saved {lang} version to {out_path}")

    print("🎉 All translations complete.")

# --- Run it ---
await main()


📦 Found 375 unique strings to translate
🌍 Translating: ...
🌍 Translating: Lesson 1: <strong>Data Science Techniques for Energy Analysi...
🌍 Translating: This chapter examines how energy systems drive and are impac...
🌍 Translating: This lesson explores how intersectionality, the interconnect...
🌍 Translating: This chapter introduces the just energy transition, highligh...
🌍 Translating: This chapter explores sustainable energy finance, market dyn...
🌍 Translating: Chapter 2: <strong>Current Status, Trends and Barriers to En...
🌍 Translating: This chapter explores strategies to support workers and comm...
🌍 Translating: Lesson 2: <strong>The Role of Data in Advancing Sustainable ...
🌍 Translating: Sustainable energy in Asia Pacific...
🌍 Translating: This chapter explores planning, monitoring, and evaluation f...
🌍 Translating: Chapter 1: <strong>Fundamentals of Energy Governance</strong...
🌍 Translating: Lesson 3: <strong>Closing key messages and call to action</s...
🌍 Translating: This

In [1]:
##translate all lessons

##Part 1: Scan files

import json
from collections import defaultdict
from pathlib import Path

# --- Compare JSON content ---
def files_differ(path1, path2) -> bool:
    try:
        with open(path1, "r", encoding="utf-8") as f1, open(path2, "r", encoding="utf-8") as f2:
            return json.load(f1) != json.load(f2)
    except Exception as e:
        print(f"⚠️ Error comparing {path1} and {path2}: {e}")
        return True  # assume changed if unreadable

# --- Scan files for translation ---
def scan_files_for_translation(retranslate=False):
    files = sorted(Path(ENGLISH_FOLDER).rglob("*.json"))
    to_translate = []
    summary = defaultdict(lambda: {"total": 0, "new": 0, "missing_outputs": 0, "changed": 0})

    for file in files:
        relative_path = file.relative_to(ENGLISH_FOLDER)
        backup_path = Path(BACKUP_FOLDER) / relative_path
        has_backup = backup_path.exists()

        outputs_exist = all((Path(OUTPUT_BASE_FOLDER) / lang / relative_path).exists() for lang in TARGET_LANGUAGES)

        is_changed = has_backup and files_differ(file, backup_path)

        needs_translation = (
            retranslate or
            not has_backup or
            is_changed or
            not outputs_exist
        )

        if needs_translation:
            to_translate.append(file)

        module_name = relative_path.parts[0]
        summary[module_name]["total"] += 1
        if not has_backup:
            summary[module_name]["new"] += 1
        if is_changed:
            summary[module_name]["changed"] += 1
        if not outputs_exist:
            summary[module_name]["missing_outputs"] += 1

    return to_translate, summary

# --- Print summary ---
def print_translation_summary(to_translate, summary):
    print("\n📋 Translation Scan Summary:\n")
    print(f"🔢 Total Lessons Needing Translation: {len(to_translate)}\n")
    for module, counts in sorted(summary.items()):
        print(f"📚 {module}: {counts['total']} lessons "
              f"(new: {counts['new']}, changed: {counts['changed']}, "
              f"missing_outputs: {counts['missing_outputs']})")


In [4]:
# --- Imports ---
import os
import json
import re
import time
import hashlib
from pathlib import Path
from collections import defaultdict
from dotenv import load_dotenv
import pandas as pd
import tiktoken
from typing import Dict, List
from openai import AsyncAzureOpenAI

# --- Load .env ---
load_dotenv()

# --- Settings ---
ENGLISH_FOLDER = "../03_Outputs/SEA_Modules/en"
BACKUP_FOLDER = "../03_Outputs/SEA_Modules/en_backup"
OUTPUT_BASE_FOLDER = "../03_Outputs/SEA_Modules"
TMP_SENTENCES_FOLDER = "../03_Outputs/Lesson_Translation/tmp_sentences"
TRANSLATIONS_CACHE_FILE = "../03_Outputs/Lesson_Translation/translations_cache.json"

TARGET_LANGUAGES = ["fr", "es", "pt", "ar", "zh", "ru"]

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

MODEL_NAME = "gpt-4o-mini"
MODEL_PRICING = {
    MODEL_NAME: {"input": 0.0005 / 1000, "output": 0.0015 / 1000}
}

# --- Globals ---
start_time = None
translated_chars = 0
total_chars_to_translate = 0

# --- Encoder ---
encoder = tiktoken.encoding_for_model("gpt-4o-mini")

# --- Utils ---
def load_json(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def estimate_tokens(text):
    return len(encoder.encode(text))

def clean_text_for_hashing(text: str) -> str:
    return re.sub(r'<[^>]+>', '', text.strip())

def hash_text(text: str) -> str:
    cleaned = clean_text_for_hashing(text)
    return hashlib.sha256(cleaned.encode("utf-8")).hexdigest()

def print_progress():
    global translated_chars, total_chars_to_translate, start_time
    if not start_time or not total_chars_to_translate:
        return
    elapsed = time.time() - start_time
    percent = (translated_chars / total_chars_to_translate) * 100
    chars_per_sec = translated_chars / elapsed if elapsed > 0 else 1e-6
    eta_seconds = (total_chars_to_translate - translated_chars) / chars_per_sec
    eta_str = time.strftime("%Hh %Mm", time.gmtime(eta_seconds))
    print(f"\r🔵 Progress: {percent:.1f}% | ETA {eta_str}", end="", flush=True)

# --- Sentence Extraction ---
def split_text_into_sentences(text: str) -> List[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]

def extract_sentences(lesson_json):
    records = []
    def walk(obj, path):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if isinstance(v, str) and (
                    k in {"label", "title", "intro", "text", "body", "description", "cta", "value", "prompt","quote","author","subtitle"} 
                    or path and path[-1] == "labels"
                ):
                    sentences = split_text_into_sentences(v)
                    for i, sentence in enumerate(sentences):
                        records.append({
                            "json_path": path + [k],
                            "sentence_idx": i,
                            "sentence_text": sentence.strip(),
                            "sentence_hash": hash_text(sentence)
                        })
                else:
                    walk(v, path + [k])
        elif isinstance(obj, list):
            for idx, item in enumerate(obj):
                walk(item, path + [idx])
    walk(lesson_json, [])
    return pd.DataFrame(records)

# --- Load Translation Cache ---
try:
    translations_cache = load_json(TRANSLATIONS_CACHE_FILE) or {}
except Exception:
    translations_cache = {}

# --- Translator Client ---
client = AsyncAzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

# --- Safe Single Sentence Translate ---
async def safe_translate(text: str) -> Dict[str, str]:
    prompt = (
      "You are a professional translator.\n"
      "Translate the TEXT from English into each of the 6 target languages.\n"
      "DO NOT leave the text in English unless it is a proper noun/title.\n"
      "Preserve HTML tags exactly (e.g., <strong>...</strong>).\n"
      "Return ONLY strict JSON with keys: fr, es, pt, ar, zh, ru.\n"
      "Each value MUST be a string (not an object, not an array).\n\n"
      "TEXT:\n"
      f"{text}"
    )
    try:

        response = await client.chat.completions.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            messages=[
                {"role": "system", "content": "You are a multilingual translator. Preserve any HTML tags and return ONLY strict JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=2500,
            response_format={"type": "json_object"},
        )


        
        content = response.choices[0].message.content.strip()
        return json.loads(content)
    except Exception as e:
        print(f"\n⚠️ Translation failed, fallback applied: {e}")
        return {lang: f"[needs translation] {text}" for lang in TARGET_LANGUAGES}

# --- Translate Sentences ---
async def translate_sentences(df_sentences):
    global translated_chars
    failed_translations = 0

    for idx, row in df_sentences.iterrows():
        h = row["sentence_hash"]
        if h in translations_cache:
            cached_translation = translations_cache[h]
            if all(not cached_translation[lang].startswith("[needs translation]") for lang in TARGET_LANGUAGES):
                continue

        translated = await safe_translate(row["sentence_text"])
        translations_cache[h] = {lang: translated.get(lang, f"[needs translation] {row['sentence_text']}") for lang in TARGET_LANGUAGES}
        translations_cache[h]["en"] = row["sentence_text"]
        translated_chars += len(row["sentence_text"])

        if any(isinstance(v, str) and v.startswith("[needs translation]") for v in translated.values()):
            failed_translations += 1

        save_json(translations_cache, TRANSLATIONS_CACHE_FILE)
        print_progress()

    return failed_translations

_PHRASE_FIX_CACHE = {}

def load_phrase_fixes(lang: str, base_dir: str | Path = "../02_Inputs/Translation"):
    """
    Loads ../02_Inputs/Translation/{lang}.xlsx with columns: original, fixed
    Returns list[tuple[original, fixed]] sorted longest-first for safer replacement.
    """
    key = (lang, str(base_dir))
    if key in _PHRASE_FIX_CACHE:
        return _PHRASE_FIX_CACHE[key]

    xlsx_path = Path(base_dir) / f"{lang}.xlsx"
    if not xlsx_path.exists():
        _PHRASE_FIX_CACHE[key] = []
        return []

    df = pd.read_excel(xlsx_path)

    # Normalize column names
    df.columns = [str(c).strip().lower() for c in df.columns]
    if "original" not in df.columns or "fixed" not in df.columns:
        raise ValueError(f"{xlsx_path} must have columns: 'original' and 'fixed'")

    df = df[["original", "fixed"]].dropna()
    df["original"] = df["original"].astype(str)
    df["fixed"] = df["fixed"].astype(str)

    # Drop empty originals
    df = df[df["original"].str.strip() != ""]

    pairs = list(df.itertuples(index=False, name=None))  # (original, fixed)

    # Longest-first to avoid partial replacements breaking longer phrases
    pairs.sort(key=lambda t: len(t[0]), reverse=True)

    _PHRASE_FIX_CACHE[key] = pairs
    return pairs

def apply_phrase_fixes(text: str, lang: str) -> str:
    if not text:
        return text

    pairs = load_phrase_fixes(lang)
    if not pairs:
        return text

    out = text
    for original, fixed in pairs:
        if original and original in out:
            print(fixed)
            out = out.replace(original, fixed)
    return out



# --- Insert Translations Back ---
def insert_translations(lesson_json, df_sentences, lang):
    df = df_sentences.copy()
    df["translated_text"] = df["sentence_hash"].map(lambda h: translations_cache[h][lang])

    grouped = df.groupby(df["json_path"].apply(lambda x: tuple(x)))

    for path, group in grouped:
        ref = lesson_json
        for k in path[:-1]:
            if isinstance(ref, list) and isinstance(k, int):
                ref = ref[k]
            elif isinstance(ref, dict):
                ref = ref.get(k, {})
        last_key = path[-1]
        merged_text = " ".join(group.sort_values("sentence_idx")["translated_text"].tolist())

        # ✅ APPLY PHRASE FIXES RIGHT BEFORE WRITING BACK
        merged_text = apply_phrase_fixes(merged_text, lang)
        
        if isinstance(ref, dict) and last_key in ref:
            ref[last_key] = merged_text

    return lesson_json

# --- Main Translation Pipeline ---
async def run_translation_pipeline(retranslate=False):
    global start_time, translated_chars, total_chars_to_translate

    to_translate, summary = scan_files_for_translation(retranslate)
    print_translation_summary(to_translate, summary)

    if not to_translate:
        print("✅ No files to process.")
        return

    start_time = time.time()
    translated_chars = 0
    total_chars_to_translate = 0

    lessons = []
    for file in to_translate:
        lesson_json = load_json(file)
        df_sentences = extract_sentences(lesson_json)
        lessons.append((file, lesson_json, df_sentences))
        total_chars_to_translate += sum(len(t) for t in df_sentences["sentence_text"])

    for file, lesson_json, df_sentences in lessons:
        print("\n")
        print(f"🔵 Translating: {Path(file).relative_to(ENGLISH_FOLDER)}")

        tmp_path = Path(TMP_SENTENCES_FOLDER) / Path(file).relative_to(ENGLISH_FOLDER).with_suffix(".csv")
        tmp_path.parent.mkdir(parents=True, exist_ok=True)
        df_sentences.to_csv(tmp_path, index=False)

        failed_translations = await translate_sentences(df_sentences)

        for lang in TARGET_LANGUAGES:
            filled = insert_translations(json.loads(json.dumps(lesson_json)), df_sentences, lang)
            out_path = Path(OUTPUT_BASE_FOLDER) / lang / Path(file).relative_to(ENGLISH_FOLDER)
            save_json(filled, out_path)

        backup_path = Path(BACKUP_FOLDER) / Path(file).relative_to(ENGLISH_FOLDER)
        save_json(lesson_json, backup_path)

        print("\n")
        if failed_translations > 0:
            print(f"🟠 Lesson translated with {failed_translations} missed.")
        else:
            print("🟢 Lesson translated.")

    elapsed = time.time() - start_time
    input_cost = translated_chars * MODEL_PRICING[MODEL_NAME]["input"]
    output_cost = translated_chars * MODEL_PRICING[MODEL_NAME]["output"]
    total_cost = input_cost + output_cost

    print("\n✅ All translations completed!")
    print(f"🕒 Time: {time.strftime('%Hh %Mm', time.gmtime(elapsed))}")
    print(f"🔢 Characters Translated: {translated_chars}")
    print(f"💲 Estimated Cost: ${total_cost:.6f}")

# --- To Run ---
await run_translation_pipeline(retranslate=True)


📋 Translation Scan Summary:

🔢 Total Lessons Needing Translation: 212

📚 Module_1: 26 lessons (new: 0, changed: 0, missing_outputs: 0)
📚 Module_2: 21 lessons (new: 0, changed: 0, missing_outputs: 0)
📚 Module_3: 23 lessons (new: 0, changed: 0, missing_outputs: 0)
📚 Module_4: 30 lessons (new: 0, changed: 0, missing_outputs: 0)
📚 Module_5: 26 lessons (new: 0, changed: 0, missing_outputs: 0)
📚 Module_6: 24 lessons (new: 0, changed: 0, missing_outputs: 0)
📚 Module_7: 24 lessons (new: 0, changed: 0, missing_outputs: 0)
📚 Module_8: 19 lessons (new: 0, changed: 0, missing_outputs: 0)
📚 Module_9: 18 lessons (new: 16, changed: 0, missing_outputs: 16)
📚 module_structure.json: 1 lessons (new: 0, changed: 1, missing_outputs: 0)


🔵 Translating: Module_1/1.-1.-1.json


🟢 Lesson translated.


🔵 Translating: Module_1/1.-1.0.json


🟢 Lesson translated.


🔵 Translating: Module_1/1.0.-1.json


🟢 Lesson translated.


🔵 Translating: Module_1/1.0.0.json


🟢 Lesson translated.


🔵 Translating: Module_1/1.1.

In [ ]:
def generate_manual_translation_files():
    print("\n📤 Generating manual translation files for untranslated content...")

    files = sorted(Path(ENGLISH_FOLDER).rglob("*.json"))
    missing_translations_by_lang = {lang: {} for lang in TARGET_LANGUAGES}
    source_lessons_by_lang = {lang: set() for lang in TARGET_LANGUAGES}

    for file in files:
        lesson_json = load_json(file)
        if lesson_json is None:
            continue

        relative_path = Path(file).relative_to(ENGLISH_FOLDER)
        df_sentences = extract_sentences(lesson_json)

        for _, row in df_sentences.iterrows():
            h = row["sentence_hash"]
            en_text = row["sentence_text"]

            if h in translations_cache:
                for lang in TARGET_LANGUAGES:
                    translation = translations_cache[h].get(lang, "")
                    if isinstance(translation, str) and "[needs translation]" in translation:
                        missing_translations_by_lang[lang][h] = {
                            "en": en_text,
                            "text": translation,
                            "source": str(relative_path)
                        }
                        source_lessons_by_lang[lang].add(str(relative_path))

    output_folder = Path("../03_Outputs/SEA_Modules/Lesson_Translation/Manual")
    output_folder.mkdir(parents=True, exist_ok=True)

    for lang, data in missing_translations_by_lang.items():
        output_file = output_folder / f"{lang}-manual.json"
        save_json(data, output_file)
        print(f"\n📄 {output_file.name} → {len(data)} entries")
        print(f"📘 Lessons with missing translations in {lang}:")
        for src in sorted(source_lessons_by_lang[lang]):
            print(f"   - {src}")

    print("\n✅ Manual translation files generated.\n")


In [ ]:
generate_manual_translation_files()